In [28]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib.gridspec as gridspec

import pandas as pd
import numpy as np

In [4]:
prob_data = pd.read_csv("data_with_probs.csv")
combined_deep = pd.read_csv("combined_deep.csv")
input_deep = pd.read_csv("input_deep.csv")
output_deep = pd.read_csv("output_deep.csv")
data_adv = pd.read_csv("data_adv.csv")

/var/folders/55/cmm1r3h92wl2408nd3911m2c0000gn/T/ipykernel_56027/935775883.py:2: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  combined_deep = pd.read_csv("combined_deep.csv")
/var/folders/55/cmm1r3h92wl2408nd3911m2c0000gn/T/ipykernel_56027/935775883.py:3: DtypeWarning: Columns (46) have mixed types. Specify dtype option on import or set low_memory=False.
  input_deep = pd.read_csv("input_deep.csv")
/var/folders/55/cmm1r3h92wl2408nd3911m2c0000gn/T/ipykernel_56027/935775883.py:4: DtypeWarning: Columns (29) have mixed types. Specify dtype option on import or set low_memory=False.
  output_deep = pd.read_csv("output_deep.csv")


In [2]:
def draw_field(ax=None, figsize = (12, 6)):
    if ax is None:
        fig, ax = plt.subplots(figsize = figsize)
    else:
        fig = ax.figure

    # Set Axis Limits
    ax.set_xlim(0, 120)
    ax.set_ylim(0, 53.3)

    # Add Rectangle for Field
    #  Rectangle((bottom_left_x, bottom_left_y), width, height)
    ax.add_patch(Rectangle((0, 0), 120, 53.3, color="#3f995b"))

    # Add Rectangles for Endzones
    ax.add_patch(Rectangle((0, 0), 10, 53.3, color="#00338D"))
    ax.add_patch(Rectangle((110, 0), 10, 53.3, color="#00338D"))

    # Yardlines for every 5 yards
    for x in range(10, 111, 5):
        if x % 10 == 0: # make 10 yard lines bold
            linewidth = 2
        else:
            linewidth = 1

        # plot([start_x, end_x], [start_y, end_y]
        ax.plot([x, x], [0, 53.3], color = "white", linewidth = linewidth, alpha = 0.8)

    # Hash Marks - 70.75 feet from each sideline
    bottom_hash = 70.75 / 3
    top_hash = 53.3 - bottom_hash
    for x in range(11, 109):
        ax.plot([x, x], [bottom_hash - 0.2, bottom_hash + 0.2], color = "white")
        ax.plot([x, x], [top_hash - 0.2, top_hash + 0.2], color = "white")

    # Yardline Numbers every 10 yards
    for x in range(20, 110, 10):
        if x <= 60:
            yardline_num = x - 10
        else:
            yardline_num = 110 - x

        # Top yardline numbers
        ax.text(x + 1, 53.3 - 5, str(yardline_num), color="white", ha="center", va="center",
            fontsize=15, fontweight="bold", rotation=270, alpha=0.8)

        # Bottom yardline numbers
        ax.text(x - 1, 5, str(yardline_num), color="white", ha="center", va="center",
            fontsize=15, fontweight="bold", rotation=90, alpha=0.8)

    # Clean up axes
    ax.set_xticks([]) # remove all x-axis ticks
    ax.set_yticks([]) # remove all y-axis ticks
    ax.set_facecolor("#3f995b") # changes background of plotting to this color
    for spine in ax.spines.values(): # remove borders around plots
        spine.set_visible(False)

    return ax

In [74]:
def animate_play(df, prob_data, game_id, play_id):
    
    # Get data for play only
    df_play = df[(df['game_id'] == game_id) & (df['play_id'] == play_id)].copy()
    prob_data = prob_data[(prob_data['game_id'] == game_id) & (prob_data['play_id'] == play_id)].copy()

    # Get list of frames in order, and number of frames
    df_play = df_play.sort_values("frame_global").copy()
    frames = df_play['frame_global'].sort_values().unique() # a list of global frame numbers

    # Merge probability onto play frames
    prob_map = dict(zip(prob_data['frame_global'], prob_data['completion_prob']))
    df_play['completion_prob'] = df_play['frame_global'].map(prob_map).fillna(0)

    # Setup figure with 2 panels
    fig = plt.figure(figsize=(16, 12))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], width_ratios=[1, 1, 1], hspace=0.1, wspace=0.03)
    ax_field = fig.add_subplot(gs[0, :])
    ax_prob = fig.add_subplot(gs[1, 1])

    fig.subplots_adjust(left=0.01, right=0.99, top=0.90, bottom=0.05)

    # Draw field on field axis
    draw_field(ax_field)
    ax_field.set_aspect('equal')

    # Ball landing point (static)
    ball_land_x = df_play['ball_land_x'].iloc[0]
    ball_land_y = df_play['ball_land_y'].iloc[0]
    ax_field.scatter(ball_land_x, ball_land_y, s=60, color='saddlebrown', edgecolor='black', label='Ball Landing')

    # Setup Completion Probability Plot
    cp_all_frames = df_play[df_play['player_role']=='Targeted Receiver']\
                        .groupby('frame_global')['completion_prob']\
                        .first()\
                        .reindex(frames, fill_value=0)
    
    prob_line, = ax_prob.plot(frames, cp_all_frames.values, color='blue', linewidth=2)
    indicator_line = ax_prob.axvline(frames[0], color='red', linestyle='--', linewidth=2)

    # Set lower x limit of probability plot to ball release
    after_frames = df_play.loc[df_play['to_release'] == 'after', 'frame_global']
    if not after_frames.empty:
        first_after_frame = after_frames.min()
    else:
        first_after_frame = frames[0]

    # Get Initial Completion Probability after Release
    hline_y = cp_all_frames.loc[first_after_frame]
    
    ax_prob.set_xlim(first_after_frame, frames[-1])
    ax_prob.set_ylim(0, 1)
    ax_prob.axhline(hline_y, color='blue', linestyle='--', linewidth=1)
    ax_prob.set_xlabel("Frame Index")
    ax_prob.set_ylabel("Completion Probability")
    ax_prob.set_title("Completion Probability Over Time")

    # Scatter Plots for Players
    player_scatter = ax_field.scatter([], [], s=60, color='white', edgecolor='black')
    rec_scatter = ax_field.scatter([], [], s=120, color='yellow', edgecolor='black')
    prob_text = ax_field.text(60, 55, "Completion Probability: 0.00", fontsize=20, color='black', ha='center')

    # Set Play Context Text to Left of Prob Plot
    # --- Get Play Context
    #row = prob_data.iloc[0]
    #print(prob_data.columns
    #complete = row['complete']
    #pos_score = pre_snap_home_score if possession_team == home_team_abbr else pre_snap_visitor_score
    #def_score = pre_snap_home_score if possession_team == visitor_team_abbr else pre_snap_visitor_score
    #def_team = home_team_abbr if possession_team == visitor_team_abbr else visitor_team_abbr

    #pos_score_text = ax_prob.text(0, 1, f"{possession_team}:{pos_score}", fontsize=14, color='black')

    def update(frame_id):
        df_frame = df_play[df_play['frame_global'] == frame_id]

        # Non-Receivers
        df_non_rec = df_frame[df_frame['player_role'] != 'Targeted Receiver']
        player_scatter.set_offsets(np.column_stack([df_non_rec['x'], df_non_rec['y']]))

        # Receiver
        df_rec = df_frame[df_frame["player_role"] == 'Targeted Receiver']
        if len(df_rec) > 0:
            rec_scatter.set_offsets(np.column_stack([df_rec["x"], df_rec["y"]]))
        else:
            rec_scatter.set_offsets(np.empty((0, 2)))

        # Update probability text and indicator line
        curr_cp = df_frame[df_frame['player_role']=='Targeted Receiver']['completion_prob'].iloc[0] \
                  if len(df_rec) > 0 else 0
        prob_text.set_text(f"Completion Probability: {curr_cp:.2f}")
        indicator_line.set_xdata([frame_id, frame_id])

        return player_scatter, rec_scatter, indicator_line, prob_text

    anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False, repeat=False)
    plt.close(fig)
    #plt.tight_layout()
    #plt.show()

    return anim

In [71]:
list(data_adv.columns)

['game_id',
 'play_id',
 'frame_global',
 'to_release',
 'home_team_abbr',
 'visitor_team_abbr',
 'play_description',
 'quarter',
 'game_clock',
 'down',
 'yards_to_go',
 'yardline_side',
 'yardline_number',
 'possession_team',
 'pre_snap_home_score',
 'pre_snap_visitor_score',
 'pass_length',
 'pass_result',
 'offense_formation',
 'receiver_alignment',
 'ball_land_x',
 'ball_land_y',
 'route_of_targeted_receiver',
 'pass_location_type',
 'team_coverage_man_zone',
 'team_coverage_type',
 'yards_gained',
 'frame_id_x',
 'tgt_rec_name',
 'tgt_rec_x',
 'tgt_rec_y',
 'tgt_rec_s',
 'tgt_rec_a',
 'tgt_rec_o',
 'tgt_rec_dir',
 'tgt_rec_ball_dist',
 'complete',
 'frame_id_y',
 'def_1_name',
 'def_2_name',
 'def_3_name',
 'def_4_name',
 'def_1_x',
 'def_2_x',
 'def_3_x',
 'def_4_x',
 'def_1_y',
 'def_2_y',
 'def_3_y',
 'def_4_y',
 'def_1_s',
 'def_2_s',
 'def_3_s',
 'def_4_s',
 'def_1_a',
 'def_2_a',
 'def_3_a',
 'def_4_a',
 'def_1_o',
 'def_2_o',
 'def_3_o',
 'def_4_o',
 'def_1_dir',
 'def_2_d

In [75]:
anim = animate_play(combined_deep, prob_data, 2023121007, 3240)
HTML(anim.to_jshtml())

In [ ]:
filt = data_adv[(data_adv['game_id'] == 2023121007) &
                     (data_adv['play_id'] == 3240)].copy()

x = filt[['frame_global', 'flight_normalized_time', 'to_release', 'tgt_rec_x', 'tgt_rec_y', 'min_ball_dist_all_def']]
x.head(60)